# Tarea 4 — Vibraciones de una barra con resorte
IMEC 4001 — Matemáticas Aplicadas  
Universidad de los Andes · 2026-2

**Estudiante:** Luis Alejandro Rodríguez


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq


## Ejercicio 2c

A partir de las condiciones de borde de los dos casos se obtiene la misma ecuación característica:

$$
(\beta L)\tan(\beta L)=\frac{kL}{EA}.
$$

Definiendo

$$
\lambda=\beta L,
\qquad
\gamma=\frac{kL}{EA},
$$

la ecuación que se debe resolver numéricamente es

$$
\boxed{F(\lambda)=\lambda\tan(\lambda)-\gamma=0}.
$$

Los valores de $\lambda_i$ corresponden a las raíces positivas de esta ecuación. Luego,

$$
\beta_i=\frac{\lambda_i}{L},
\qquad
\alpha_i=\beta_i^2.
$$


### Parámetros

Se consideran

$$
L=1,\qquad \rho=1,
$$

y los valores indicados para el parámetro $kL/(EA)$:

$$
\frac{kL}{EA}\in\{0.1,\,1,\,10\}.
$$


In [ ]:
# Parámetros
L = 1
rho = 1

parametros = [0.1, 1, 10]
n_modos = 3

display(f"L = {L}")
display(f"rho = {rho}")
display(f"Parametros kL/(EA) = {parametros}")


### Función característica

Para cada valor de $kL/(EA)$ se define

$$
F(\lambda)=\lambda\tan(\lambda)-\frac{kL}{EA}.
$$

Los primeros tres ceros positivos se buscan en los intervalos

$$
(0,\pi/2),\qquad
(\pi,3\pi/2),\qquad
(2\pi,5\pi/2).
$$

Se utiliza `brentq`, que encuentra una raíz de una función no lineal dentro de un intervalo donde la función cambia de signo.


In [ ]:
def funcion(lambda_, gamma):
    return lambda_ * np.tan(lambda_) - gamma


def calcular_raices(gamma, n_modos=3):
    raices = []

    for n in range(n_modos):
        a = n * np.pi + 1e-10
        b = n * np.pi + np.pi / 2 - 1e-10

        raiz = brentq(funcion, a, b, args=(gamma,))
        raices.append(raiz)

    return np.array(raices)


In [ ]:
# Cálculo de los tres primeros parámetros lambda_i

resultados = {}

for gamma in parametros:
    lambdas = calcular_raices(gamma, n_modos)
    resultados[gamma] = lambdas

    display(f"kL/(EA) = {gamma}")
    for i, value in enumerate(lambdas, start=1):
        display(f"lambda_{i} = {value:.6f}")


### Resultados

Los valores obtenidos numéricamente son los tres primeros ceros positivos de $F(\lambda)$ para cada valor del parámetro.


In [ ]:
print(f"{'kL/(EA)':>10} {'lambda_1':>12} {'lambda_2':>12} {'lambda_3':>12}")
print("-" * 50)

for gamma in parametros:
    lambdas = resultados[gamma]
    print(
        f"{gamma:10.1f}"
        f"{lambdas[0]:12.6f}"
        f"{lambdas[1]:12.6f}"
        f"{lambdas[2]:12.6f}"
    )


### Verificación gráfica

Se grafica $F(\lambda)$ para cada parámetro y se marcan las tres raíces encontradas por el algoritmo.


In [ ]:
for gamma in parametros:
    lambdas = resultados[gamma]

    x_max = 2.5 * np.pi
    x = np.linspace(0, x_max, 5000)
    y = funcion(x, gamma)

    # Evitar valores muy grandes cerca de las asíntotas
    y[np.abs(y) > 20] = np.nan

    plt.figure(figsize=(9, 5))
    plt.plot(x, y, label=fr"$F(\lambda)$, $kL/(EA)={gamma}$")
    plt.axhline(0, linewidth=1)
    plt.scatter(
        lambdas,
        np.zeros_like(lambdas),
        zorder=3,
        label="Raíces"
    )

    plt.xlabel(r"$\lambda$")
    plt.ylabel(r"$F(\lambda)$")
    plt.title(fr"Raíces de $F(\lambda)$ para $kL/(EA)={gamma}$")
    plt.grid()
    plt.legend()
    plt.show()


### Cálculo de los autovalores $\alpha_i$

Como

$$
\lambda_i=\beta_iL,
$$

se tiene

$$
\beta_i=\frac{\lambda_i}{L}
$$

y por tanto

$$
\boxed{\alpha_i=\beta_i^2=\left(\frac{\lambda_i}{L}\right)^2}.
$$

Para $L=1$, resulta simplemente $\alpha_i=\lambda_i^2$.


In [ ]:
# Autovalores alpha_i

for gamma in parametros:
    lambdas = resultados[gamma]
    alphas = (lambdas / L) ** 2

    display(f"kL/(EA) = {gamma}")
    for i, value in enumerate(alphas, start=1):
        display(f"alpha_{i} = {value:.6f}")


### Conclusión

El algoritmo numérico permite encontrar los parámetros $\lambda_i$ que satisfacen la ecuación característica. Los dos escenarios físicos tienen la misma ecuación característica, por lo que producen los mismos autovalores. La diferencia entre los casos aparece en la forma de los modos espaciales $u_i(x)$, debido a la ubicación del resorte y del extremo libre.
